In [1]:
from ml_recon.Models.fastMRI_unet import Unet
from torch.utils.data import DataLoader
from ml_recon.Transforms import (pad, trim_coils, combine_coil, toTensor, permute, 
                                   view_as_real, remove_slice_dim, fft_2d, norm_normalize, pad_recon)
from ml_recon.Dataset.undersampled_dataset import UndersampledKSpaceDataset
from torchvision.transforms import Compose
import numpy as np
import torch
from ml_recon.Utils.collate_function import collate_fn
from datetime import datetime

In [2]:
torch.manual_seed(0)
np.random.seed(0)

In [3]:
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter('/tmp/kadota_runs/' +  datetime.now().strftime("%Y%m%d-%H%M%S"))

In [4]:
%load_ext autoreload
%autoreload 2 

In [5]:
transforms = Compose(
    (
        trim_coils(12),
        pad((640, 320)),
        pad_recon((320, 320)), 
        fft_2d(axes=[2,3]),
        combine_coil(),
        toTensor(),
        #norm_normalize(),
        view_as_real(),
        permute() 
    )
)
dataset = UndersampledKSpaceDataset('/home/kadotab/projects/def-mchiew/kadotab/Datasets/fastMRI/multicoil_train', transforms=transforms, R=4)
dataloader = DataLoader(dataset, batch_size=1, collate_fn=collate_fn)
    

In [6]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [7]:
model = Unet(2, 2)
model.to(device)

Unet(
  (down_sample_layers): ModuleList(
    (0): ConvBlock(
      (layers): Sequential(
        (0): Conv2d(2, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
        (2): LeakyReLU(negative_slope=0.2, inplace=True)
        (3): Dropout2d(p=0.0, inplace=False)
        (4): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (5): InstanceNorm2d(32, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
        (6): LeakyReLU(negative_slope=0.2, inplace=True)
        (7): Dropout2d(p=0.0, inplace=False)
      )
    )
    (1): ConvBlock(
      (layers): Sequential(
        (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (1): InstanceNorm2d(64, eps=1e-05, momentum=0.1, affine=False, track_running_stats=False)
        (2): LeakyReLU(negative_slope=0.2, inplace=True)
        (3): Dropout2

In [8]:
loss_fn = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

In [9]:
def train(model, loss_function, optimizer, dataloader, epoch=7):
    cur_loss = 0
    current_index = 0
    try:
        for e in range(epoch):
            for data in dataloader:
                undersampled = data['undersampled']
                recon = data['recon']
                for i in range(undersampled.shape[0]):
                    optimizer.zero_grad()

                    recon_slice = torch.flip(recon[[i], ...].float(), (1, ))
                    undersampled_slice = undersampled[[i],...].float()

                    recon_slice = recon_slice.to(device)
                    undersampled_slice = undersampled_slice.to(device)
    
                    predicted_sampled = model(undersampled_slice)

                    predicted_sampled = torch.sqrt(predicted_sampled.sum(1).pow(2) + 1e-6)
                    assert not predicted_sampled.isnan().any()
                    predicted_sampled = predicted_sampled[:, 160:-160, :]
                    loss = loss_function(predicted_sampled, recon_slice)
                    
                    loss.backward()
                    optimizer.step()
                    
                    cur_loss += loss.item()
                    current_index += 1
                    if current_index % 100 == 99:
                        writer.add_histogram('sens/weights1', next(model.down_sample_layers[0].conv1.parameters()), current_index)
                        writer.add_histogram('castcade0/weights1', next(model.down_sample_layers[0].conv2.parameters()), current_index)
                        writer.add_histogram('castcade0/weights2', next(model.down_sample_layers[3].conv.conv1.parameters()), current_index)
                        writer.add_histogram('castcade0/weights11', next(model.up_sample_layers[3].conv.conv2.parameters()), current_index)
                        writer.add_histogram('castcade0/weights12', next(model.conv1d.parameters()), current_index)
                        writer.add_scalar('Loss/train', cur_loss, current_index)
                        print(f"Iteration: {current_index + 1:>d}, Loss: {cur_loss:>7f}")
                    cur_loss = 0
    except KeyboardInterrupt:
        pass

    model_name = model.__class__.__name__
    date = datetime.now().strftime("%Y%m%d-%H%M%S")
    torch.save({
        model: model.state_dict(), 
        optimizer: optimizer.state_dict()
        }, '.ml_recon/Model_Weights/' + date + model_name + '.pt')

In [10]:
train(model, loss_fn, optimizer, dataloader)

/tmp/ipykernel_11998/1975667055.py:5: UserWarning: Anomaly Detection has been enabled. This mode will increase the runtime and should only be enabled for debugging.
  with torch.autograd.detect_anomaly(True):


RuntimeError: Parent directory .ml_recon/Model_Weights does not exist.